In [23]:
import pandas as pd
import plotly.express as px
from pathlib import Path

# --- 1) Charger le CSV ---
BASE_DIR = Path().resolve()  # this is 04-Maps
csv_path = BASE_DIR.parent / "02-Get weather" / "sites_weather_summary.csv"

df_summary = pd.read_csv(csv_path)

# --- 2) Préparer les données pour Plotly ---
# Forcer les colonnes numériques
for col in ["avg_temp_c", "expected_rain_mm", "nice_score", "lat", "lon"]:
    df_summary[col] = pd.to_numeric(df_summary[col], errors="coerce")

# Garder seulement les sites valides, classer par nice_score et prendre le Top 5
df_top = (
    df_summary
    .dropna(subset=["nice_score", "lat", "lon", "avg_temp_c"])
    .sort_values("nice_score", ascending=False)
    .head(5)
    .copy()
)

if df_top.empty:
    print("Pas de sites avec données météo valides.")
else:
    # Taille positive pour l'affichage
    df_top["size_marker"] = (df_top["nice_score"] - df_top["nice_score"].min()) + 1

# --- 3) Carte Plotly ---
fig = px.scatter_map(
    df_top,
    lat="lat",
    lon="lon",
    size="size_marker",
    color="avg_temp_c",
    color_continuous_scale=[(0, "blue"), (0.5, "purple"), (1, "red")],
    hover_name="site",
    hover_data={
        "avg_temp_c": True,
        "expected_rain_mm": True,
        "nice_score": True,
        "lat": False, "lon": False
    },
    map_style="open-street-map",
)

fig.update_layout(
    map=dict(
        center={"lat": 46.6, "lon": 2.4},
        zoom=4.8,
    ),
    title="Top 5 destinations (nice_score)",
    margin=dict(r=0, l=0, t=40, b=0),
    height=700,
    width=700,
)

fig.show()
